In [1]:
import sys
import boto3
import pyarrow
import pandas as pd
import pyarrow.parquet as pq

MINIO_ENDPOINT = "http://minio:9000"  # DENTRO do container
ACCESS_KEY = "minio"
SECRET_KEY = "minio123"

import pyarrow.fs as fs

fs_s3 = fs.S3FileSystem(
    endpoint_override=MINIO_ENDPOINT,
    access_key=ACCESS_KEY,
    secret_key=SECRET_KEY,
    scheme="http"
)
fs_s3.get_file_info(fs.FileSelector(""))

[<FileInfo for 'bronze': type=FileType.Directory>,
 <FileInfo for 'gold': type=FileType.Directory>,
 <FileInfo for 'silver': type=FileType.Directory>]

In [2]:
def load_parquet(path):
    return pq.ParquetDataset(
        path,
        filesystem=fs_s3
    ).read_pandas().to_pandas()

## carregar as bases
df_sinistros = load_parquet("silver/infosiga/sinistros_2015-2021/dt=2026-04-25/sinistros_2015-2021.parquet")

df_pessoas = load_parquet("silver/infosiga/pessoas_2015-2021/dt=2026-04-25/pessoas_2015-2021.parquet")

df_veiculos = load_parquet("silver/infosiga/veiculos_2015-2021/dt=2026-04-25/veiculos_2015-2021.parquet")

df_pessoas.shape, df_veiculos.shape, df_sinistros.shape

((475178, 29), (633113, 12), (535566, 48))

((475178, 29), (633113, 12), (535566, 48))

In [3]:
df_sinistros.head(3)

,id_sinistro,tipo_registro,data_sinistro,ano_sinistro,mes_sinistro,dia_sinistro,hora_sinistro,ano_mes_sinistro,dia_da_semana,turno,...,tp_sinistro_colisao_traseira,tp_sinistro_colisao_lateral,tp_sinistro_colisao_transversal,tp_sinistro_colisao_outros,tp_sinistro_choque,tp_sinistro_capotamento,tp_sinistro_engavetamento,tp_sinistro_tombamento,tp_sinistro_outros,tp_sinistro_nao_disponivel
0,2501575,SINISTRO FATAL,21/12/2014,2014,12,21,20:00,2014/12,Domingo,NOITE,...,NaN,None,None,None,None,None,None,None,None,None
1,2462674,SINISTRO FATAL,31/12/2014,2014,12,31,22:53,2014/12,Quarta-feira,NOITE,...,NaN,None,None,None,None,None,None,None,None,None
2,2470843,SINISTRO FATAL,01/01/2015,2015,1,1,16:00,2015/01,Quinta-feira,TARDE,...,NaN,None,None,None,None,S,None,None,None,None


In [15]:
df_pessoas.head(3)

,id_sinistro,id_veiculo,cod_ibge,municipio,regiao_administrativa,tipo_via,tipo_veiculo_vitima,sexo,idade,gravidade_lesao,...,dia_sinistro,ano_mes_sinistro,data_obito,ano_obito,mes_obito,dia_obito,ano_mes_obito,local_obito,local_via,tempo_sinistro_obito
0,2473612,NaN,3509957,CANAS,SÃO JOSÉ DOS CAMPOS,ESTRADAS E RODOVIAS,AUTOMOVEL,MASCULINO,24.0,FATAL,...,1,2015/01,01/01/2015,2015.0,1.0,1.0,2015/01,VIA,PUBLICO,0.0
1,2460453,NaN,3543402,RIBEIRAO PRETO,RIBEIRÃO PRETO,VIAS URBANAS,None,MASCULINO,68.0,FATAL,...,3,2015/01,03/01/2015,2015.0,1.0,3.0,2015/01,ESTABELECIMENTO DE SAUDE,PUBLICO,0.0
2,2457878,NaN,3505500,BARRETOS,BARRETOS,ESTRADAS E RODOVIAS,CAMINHAO,FEMININO,54.0,FATAL,...,5,2015/01,05/01/2015,2015.0,1.0,5.0,2015/01,VIA,PUBLICO,0.0


In [16]:
df_veiculos.head(3)

,id_sinistro,id_veiculo,marca_modelo,ano_fab,ano_modelo,cor_veiculo,tipo_veiculo,data_sinistro,ano_sinistro,mes_sinistro,dia_sinistro,ano_mes_sinistro
0,2501575,72104,None,NaN,NaN,None,OUTROS,21/12/2014,2014,12,21,2014/12
1,2456933,2281029,None,NaN,NaN,None,OUTROS,23/12/2014,2014,12,23,2014/12
2,2487781,50348,None,NaN,NaN,None,OUTROS,28/12/2014,2014,12,28,2014/12


In [4]:
df_gold_tempo = (
    df_sinistros
    .groupby(["ano_sinistro", "mes_sinistro", "turno"])
    .agg(
        total_sinistros=("id_sinistro", "count")
    )
    .reset_index()
)


In [5]:
df_gold_tempo.head()

,ano_sinistro,mes_sinistro,turno,total_sinistros
0,2014,12,MANHA,1
1,2014,12,NAO DISPONIVEL,3
2,2014,12,NOITE,5
3,2014,12,TARDE,2
4,2015,1,MADRUGADA,112
